# Galaxy-X-os  —  One-Click Colab Pipeline (SCALE x ODYSSEY)

Classify raw astronomical images into **5 celestial categories** with **Ensemble: ConvNeXt-Base + Swin-B + EfficientNet-B3**.

**How to run:** `Runtime → Change runtime type → GPU (T4)`, then `Runtime → Run all`.

Pipeline: clone → install → prepare data → train (3 backbones) → evaluate ensemble → Grad-CAM → download results.

## Cell 1 — Clone repo + install dependencies

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Srujan0798/Galaxy-X-os"
REPO_DIR = "/content/Galaxy-X-os"

# Clone if missing; otherwise pull latest
if os.path.exists(os.path.join(REPO_DIR, "src/prepare_data.py")):
    print("Repo present -> pulling latest main...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=False)
elif os.path.exists("src/prepare_data.py"):
    REPO_DIR = os.getcwd()
    print(f"Inside repo at {REPO_DIR}")
else:
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

# Install deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "astroNN", "kagglehub", "h5py"], check=False)
print("Dependencies installed.")

## Cell 1b — Verify GPU (required for training)

Training on CPU is impractically slow. Set `Runtime → Change runtime type → Hardware accelerator → GPU (T4)` before running.

In [ ]:
import torch

if not torch.cuda.is_available():
    print("WARNING: No GPU detected. Training will be very slow on CPU.")
    print("  Set Runtime -> Change runtime type -> GPU (T4) and re-run.")
    print("  Continuing in CPU demo mode (3 epochs)...")
    device = "cpu"
else:
    device = "cuda"
print(f"Using device: {device}")

## Cell 2 — Kaggle token (optional)

Spiral & elliptical use real Galaxy10 (no key). Nebula / star_cluster / planetary need a Kaggle token for real data, or fall back to procedural images.

In [ ]:
# Optional Kaggle token
KAGGLE_API_TOKEN = ""  # paste your KGAT_ token here, or leave empty

if not KAGGLE_API_TOKEN:
    try:
        from google.colab import userdata
        KAGGLE_API_TOKEN = userdata.get("KAGGLE_API_TOKEN") or ""
    except Exception:
        pass

if KAGGLE_API_TOKEN:
    os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN.strip()
    kdir = Path.home() / ".kaggle"
    kdir.mkdir(exist_ok=True)
    (kdir / "access_token").write_text(KAGGLE_API_TOKEN.strip() + "\n")
    os.chmod(kdir / "access_token", 0o600)
    print("Kaggle token installed -> REAL nebula/cluster/planetary will be attempted.")
else:
    print("No Kaggle token -> procedural fallback for nebula/cluster/planetary.")

## Cell 3 — Prepare data (real-first, safe-fallback, idempotent)

In [ ]:
!python src/prepare_data.py --per-class 500

import json
with open("data/processed/DATA_MANIFEST.json") as f:
    print(json.dumps(json.load(f), indent=2))

## Cell 4 — Train EfficientNet-B3 (standalone)

Writes `checkpoints/best_model_efficientnet_b3.pth`. On GPU: 50 epochs (~20 min). On CPU: 3 epochs (~1 hr).

In [ ]:
!python src/train.py --backbone efficientnet_b3 --checkpoint checkpoints/best_model_efficientnet_b3.pth --epochs 50 --lr 3e-4 --batch-size 32 --label-smoothing 0.1 --focal-gamma 2.0

import torch as _torch
ckpt = _torch.load("checkpoints/best_model_efficientnet_b3.pth", map_location="cpu", weights_only=True)
bva = ckpt.get("best_val_acc", None)
if bva is not None:
    print(f"EfficientNet-B3: {ckpt.get('epoch', '?')} epochs, best_val_acc={bva:.4f}")
else:
    print("Checkpoint not found or incomplete.")

## Cell 4b — Train ConvNeXt-Base (88M params)

Second backbone for ensemble. 3 epochs on CPU / 50 on GPU.

In [ ]:
!python src/train.py --backbone convnext_base --checkpoint checkpoints/best_model_convnext_base.pth --epochs 3 --lr 3e-4 --batch-size 32 --label-smoothing 0.1 --focal-gamma 2.0

import torch as _torch
ckpt = _torch.load("checkpoints/best_model_convnext_base.pth", map_location="cpu", weights_only=True)
bva = ckpt.get("best_val_acc", None)
if bva is not None:
    print(f"ConvNeXt-Base: {ckpt.get('epoch', '?')} epochs, best_val_acc={bva:.4f}")
else:
    print("Checkpoint not found or incomplete.")

## Cell 4c — Train Swin-B (88M params)

Third backbone for ensemble. 3 epochs on CPU / 50 on GPU.

In [ ]:
!python src/train.py --backbone swin_base_patch4_window7_224 --checkpoint checkpoints/best_model_swin_base_patch4_window7_224.pth --epochs 3 --lr 3e-4 --batch-size 32 --label-smoothing 0.1 --focal-gamma 2.0

import torch as _torch
ckpt = _torch.load("checkpoints/best_model_swin_base_patch4_window7_224.pth", map_location="cpu", weights_only=True)
bva = ckpt.get("best_val_acc", None)
if bva is not None:
    print(f"Swin-B: {ckpt.get('epoch', '?')} epochs, best_val_acc={bva:.4f}")
else:
    print("Checkpoint not found or incomplete.")

## Cell 5 — Evaluate (standard + TTA + Ensemble)

In [ ]:
# Single model evaluation
!python src/evaluate.py --tta standard

# Ensemble evaluation (if all 3 checkpoints exist)
import os
if all(os.path.exists(f"checkpoints/{m}.pth") for m in ["convnext_base", "swin_base", "efficientnet_b3"]):
    !python src/evaluate.py --ensemble --tta advanced --overwrite
    print("Ensemble evaluation complete!")
else:
    print("Ensemble checkpoints not all present, skipping ensemble eval.")

# Display results
import json
from IPython.display import Image as IPyImage, display
with open("results/evaluation_results.json") as f:
    results = json.load(f)
print(f"\nStandard: Acc={results['standard']['accuracy']:.4f} F1={results['standard']['macro_f1']:.4f}")
if results.get("tta"):
    print(f"TTA:      Acc={results['tta']['accuracy']:.4f} F1={results['tta']['macro_f1']:.4f}")
if results.get("uncertainty"):
    print(f"Uncertainty: epistemic={results['uncertainty']['mean_epistemic']:.6f} aleatoric={results['uncertainty']['mean_aleatoric']:.6f}")
display(IPyImage(filename="results/confusion_matrix.png"))
display(IPyImage(filename="results/per_class_metrics.png"))

## Cell 6 — Grad-CAM from the trained model

In [ ]:
!python src/gradcam.py

import glob
from IPython.display import Image as IPyImage, display
cams = sorted(glob.glob("results/gradcam/*.png"))
print(f"Generated {len(cams)} Grad-CAM images.")
for p in cams[:4]:
    display(IPyImage(filename=p))

## Cell 7 — Zip results + checkpoint and download

In [ ]:
import zipfile, os

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk("results"):
        for fn in fnames:
            fp = os.path.join(root, fn)
            z.write(fp, fp)
    for ckpt in ["best_model_efficientnet_b3.pth", "best_model_convnext_base.pth", "best_model_swin_base_patch4_window7_224.pth"]:
        if os.path.exists(f"checkpoints/{ckpt}"):
            z.write(f"checkpoints/{ckpt}", f"checkpoints/{ckpt}")

print("Wrote results.zip:", round(os.path.getsize("results.zip") / 1e6, 1), "MB")
print("\nAfter download: unzip results.zip into repo root, then commit.")

try:
    from google.colab import files
    files.download("results.zip")
except Exception as e:
    print(f"(auto-download unavailable: {e} — download results.zip from the Files panel)")